In [10]:
%pip install -r ../../requirements.txt

import pandas as pd

  Using cached scikit_learn-1.9.0-cp314-cp314-macosx_12_0_arm64.whl.metadata (11 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 19.1 MB/s  0:00:009.3 MB/s eta 0:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.4/20.4 MB 22.8 MB/s  0:00:003.4 MB/s eta 0:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [scikit-learn]0m 4/5 [scikit-learn]
Note: you may need to restart the kernel to use updated packages.


In [114]:
toronto_data = pd.read_csv('../data/toronto_real_estate_public_area.csv')

In [115]:
toronto_data.head(3)

,property_id,period,home_type_bucket,status,last_status,home_type,price,list_price,sold_price,list_date,neighbourhood,latitude,longitude,bedrooms,bedrooms_plus,bathrooms,brokerage,images_count,estimated_area_sqft
0,1,sale,CONDO,ACTIVE,PRICE_CHANGE,CONDO,388000.0,388000.0,NaN,2026-04-12T13:29:08.000Z,Waterfront Communities The Island,43.648879,-79.388044,1.0,NaN,1.0,HC REALTY GROUP INC.,46,350.0
1,2,sale,CONDO,ACTIVE,NEW,CONDO,385000.0,385000.0,NaN,2026-05-28T11:31:44.000Z,Moss Park,43.653313,-79.359882,NaN,NaN,1.0,BAKER REAL ESTATE INCORPORATED,24,350.0
2,3,sale,CONDO,ACTIVE,NEW,CONDO,299000.0,299000.0,NaN,2026-06-09T10:35:34.000Z,Church-Yonge Corridor,43.657375,-79.374087,NaN,NaN,1.0,SUPERSTARS REALTY LTD.,25,350.0


In [116]:
# We'll be using sold_price as the target for this model, and won't be using price or list_price as features.

toronto_data = toronto_data.drop(columns=['property_id', 'period', 'brokerage', 'home_type_bucket', 'price', 'list_price', 'images_count','status'])

In [117]:
toronto_data

,last_status,home_type,sold_price,list_date,neighbourhood,latitude,longitude,bedrooms,bedrooms_plus,bathrooms,estimated_area_sqft
0,PRICE_CHANGE,CONDO,NaN,2026-04-12T13:29:08.000Z,Waterfront Communities The Island,43.648879,-79.388044,1.0,NaN,1.0,350.0
1,NEW,CONDO,NaN,2026-05-28T11:31:44.000Z,Moss Park,43.653313,-79.359882,NaN,NaN,1.0,350.0
2,NEW,CONDO,NaN,2026-06-09T10:35:34.000Z,Church-Yonge Corridor,43.657375,-79.374087,NaN,NaN,1.0,350.0
3,NEW,CONDO,NaN,2026-06-20T09:57:05.000Z,Bay Street Corridor,43.661136,-79.385605,1.0,NaN,1.0,650.0
4,NEW,CONDO,NaN,2026-06-03T21:38:51.000Z,South Riverdale,43.657640,-79.350917,1.0,NaN,1.0,550.0
...,...,...,...,...,...,...,...,...,...,...,...
9699,SOLD,CONDO,555000.0,2024-03-15T16:55:50.000Z,Waterfront Communities C1,43.645958,-79.389873,1.0,1.0,1.0,550.0
9700,SOLD,CONDO,585000.0,2024-05-10T16:13:24.000Z,Little Portugal,43.642018,-79.423509,2.0,NaN,2.0,550.0
9701,SOLD,CONDO,600000.0,2024-05-15T16:08:20.000Z,Clairlea Birchmount,43.716540,-79.284514,2.0,NaN,2.0,950.0
9702,SOLD,CONDO,595000.0,2024-04-01T16:00:47.000Z,Moss Park,43.656969,-79.374122,1.0,1.0,1.0,550.0


In [118]:
tud = toronto_data['sold_price'].isnull().sum()
tud

np.int64(3244)

In [119]:
toronto_data.isnull().sum()

last_status               0
home_type                 0
sold_price             3244
list_date                 0
neighbourhood             1
latitude                  0
longitude                 0
bedrooms                290
bedrooms_plus          5460
bathrooms                50
estimated_area_sqft     162
dtype: int64

In [133]:
toronto_data['locality'].isnull().sum()

np.int64(1)

In [127]:
# Renamed column to locality after normalization

toronto_data['locality'] = (
    toronto_data['neighbourhood']
    .dropna()
    .str.lower()
    .str.replace(r'^[a-z0-9]+ - ', '', regex=True)
    .str.replace(r'[-]', ' ', regex=True)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)

In [132]:
len(toronto_data['locality'].unique())

131

In [123]:
(toronto_data['locality'].value_counts()).sum()

np.int64(9703)